In [1]:
#!/usr/bin/env python
# =============================================================================
# HAHA 2026 — Fine-tuning v3.1 (fix OOM)
#
# FIX OOM so với v3:
#   [OOM-1] Không giữ fold_models trong RAM — predict + lưu probs xong là del ngay
#   [OOM-2] predict_one_model() — predict từng model 1 trên CPU-offloaded state,
#           không bao giờ có 2 models trên GPU cùng lúc
#   [OOM-3] batch_size T2 giảm xuống 8 (xlm-roberta lớn hơn BETO)
#   [OOM-4] gradient_checkpointing cho xlm-roberta để giảm activation memory
#   [OOM-5] del model + gc.collect() + cuda.empty_cache() sau mỗi fold
#   [OOM-6] Lưu best checkpoint ra disk thay vì copy.deepcopy() state_dict to RAM
#
# CHIẾN LƯỢC (giữ nguyên từ v3):
#   - Train trên trial + dev_gold (564 T1, 362 T2)
#   - 5-fold StratifiedKFold — val ~110/~72 samples mỗi fold
#   - T1: dccuchile/bert-base-spanish-wwm-cased
#   - T2: xlm-roberta-base
#   - T2 text: thêm [LONG]/[SHORT] hint (machine dài ~25t, human ngắn ~13t)
# =============================================================================

import gc
import copy
import os
import re
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import StratifiedKFold
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)

# =============================================================================
# CONFIG
# =============================================================================
CFG = {
    "data_dir":   "/kaggle/input/datasets/ngqucnam/haha-dataset",
    "test_dir":   "/kaggle/input/datasets/ngqucnam/haha2026-test",
    "output_dir": "/kaggle/working",
    "ckpt_dir":   "/kaggle/working/ckpts",   # lưu checkpoint ra disk

    "model_t1": "dccuchile/bert-base-spanish-wwm-cased",
    "model_t2": "xlm-roberta-base",

    "folds":       5,
    "seed":        42,
    "num_workers": 0,

    "t1": {
        "max_len":               128,
        "batch_size":            16,   # BETO nhỏ, OK
        "lr":                    2e-5,
        "epochs":                5,
        "warmup_ratio":          0.1,
        "weight_decay":          0.01,
        "dropout":               0.1,
        "grad_checkpoint":       False,
    },
    "t2": {
        "max_len":               128,
        "batch_size":            8,    # [OOM-3] giảm từ 16 xuống 8
        "lr":                    2e-5,
        "epochs":                6,
        "warmup_ratio":          0.1,
        "weight_decay":          0.01,
        "dropout":               0.1,
        "grad_checkpoint":       True,  # [OOM-4] bật cho xlm-roberta
    },
}


# =============================================================================
# 1. Helpers
# =============================================================================
def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def free_gpu(*objs):
    """[OOM-5] Xóa objects, gọi gc + empty_cache."""
    for o in objs:
        del o
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def clean_text(text):
    if not isinstance(text, str):
        return ""
    return re.sub(r"\s+", " ", text).strip()


def gpu_mem_str():
    if not torch.cuda.is_available():
        return ""
    free  = torch.cuda.mem_get_info()[0] / 1024**3
    total = torch.cuda.mem_get_info()[1] / 1024**3
    return f"[GPU {total-free:.1f}/{total:.1f}GB]"


# =============================================================================
# 2. Text builders
# =============================================================================
def make_t1_text(row):
    headline = clean_text(str(row.get("headline", "")))
    context  = clean_text(str(row.get("context",  "")))
    if context and context.lower() != "nan":
        return headline + " " + context
    return headline


def make_t2_text(row):
    """
    Thêm [LONG]/[SHORT] hint: machine jokes dài (~25 từ), human ngắn (~13 từ).
    Threshold = 18 từ từ phân tích dev_gold.
    """
    headline = clean_text(str(row.get("headline", "")))
    joke     = clean_text(str(row.get("joke",     "")))
    hint     = "[LONG]" if len(joke.split()) >= 18 else "[SHORT]"
    return headline + " [SEP] " + hint + " " + joke


# =============================================================================
# 3. Dataset
# =============================================================================
class HahaDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_len,
            padding="max_length",
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


# =============================================================================
# 4. Train 1 fold — trả về ckpt_path thay vì model object
# =============================================================================
def train_fold(
    model_name, ckpt_path,
    tr_texts, tr_labels,
    val_texts, val_labels,
    cfg_task, pos_idx,
    device, use_fp16, fold_num,
):
    """
    [OOM-1] Không return model — lưu best state_dict ra disk rồi del model.
    Trả về (best_val_f1, ckpt_path).
    """
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    tr_ds  = HahaDataset(tr_texts,  tr_labels,  tokenizer, cfg_task["max_len"])
    val_ds = HahaDataset(val_texts, val_labels, tokenizer, cfg_task["max_len"])

    tr_loader  = DataLoader(tr_ds,  batch_size=cfg_task["batch_size"],
                            shuffle=True,  num_workers=CFG["num_workers"],
                            pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=cfg_task["batch_size"] * 2,
                            shuffle=False, num_workers=CFG["num_workers"],
                            pin_memory=True)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2,
        hidden_dropout_prob=cfg_task["dropout"],
        attention_probs_dropout_prob=cfg_task["dropout"],
        ignore_mismatched_sizes=True,
    ).to(device)

    # [OOM-4] Gradient checkpointing — giảm activation memory ~40%
    if cfg_task.get("grad_checkpoint", False):
        model.gradient_checkpointing_enable()
        print(f"  [fold {fold_num}] gradient_checkpointing ON")

    optimizer   = AdamW(model.parameters(),
                        lr=cfg_task["lr"], weight_decay=cfg_task["weight_decay"])
    total_steps = len(tr_loader) * cfg_task["epochs"]
    scheduler   = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(total_steps * cfg_task["warmup_ratio"]),
        num_training_steps=total_steps,
    )
    criterion = nn.CrossEntropyLoss()

    # Scaler
    scaler = None
    if use_fp16:
        try:
            scaler = torch.amp.GradScaler("cuda")
        except Exception:
            from torch.cuda.amp import GradScaler
            scaler = GradScaler()

    best_f1 = -1.0

    for epoch in range(1, cfg_task["epochs"] + 1):
        # ── Train ──
        model.train()
        total_loss = 0.0
        for batch in tqdm(tr_loader,
                          desc=f"  F{fold_num}E{epoch:02d} {gpu_mem_str()}",
                          leave=False):
            batch = {k: v.to(device) for k, v in batch.items()}
            optimizer.zero_grad(set_to_none=True)

            if use_fp16 and scaler is not None:
                try:
                    ctx = torch.amp.autocast("cuda")
                except Exception:
                    from torch.cuda.amp import autocast
                    ctx = autocast()
                with ctx:
                    out  = model(**{k: v for k, v in batch.items() if k != "labels"})
                    loss = criterion(out.logits, batch["labels"])
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                out  = model(**{k: v for k, v in batch.items() if k != "labels"})
                loss = criterion(out.logits, batch["labels"])
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            scheduler.step()
            total_loss += loss.item()

        # ── Validate ──
        model.eval()
        preds, targets = [], []
        with torch.no_grad():
            for batch in val_loader:
                batch  = {k: v.to(device) for k, v in batch.items()}
                logits = model(**{k: v for k, v in batch.items() if k != "labels"}).logits
                preds.extend(logits.argmax(-1).cpu().tolist())
                targets.extend(batch["labels"].cpu().tolist())

        val_f1 = f1_score(targets, preds, pos_label=pos_idx,
                          average="binary", zero_division=0)
        mark   = " ✓" if val_f1 > best_f1 else ""
        print(f"  F{fold_num} E{epoch:02d}  "
              f"loss={total_loss/len(tr_loader):.4f}  val_F1={val_f1:.4f}{mark}  "
              f"{gpu_mem_str()}")

        if val_f1 > best_f1:
            best_f1 = val_f1
            # [OOM-6] Lưu ra disk ngay, không giữ trong RAM
            torch.save(model.state_dict(), ckpt_path)

    if not Path(ckpt_path).exists():
        # Nếu val_f1 = 0 suốt (edge case), lưu epoch cuối
        torch.save(model.state_dict(), ckpt_path)

    # [OOM-5] Del model ngay sau khi lưu
    free_gpu(model)
    print(f"  → Fold {fold_num} best F1={best_f1:.4f}  ckpt saved  {gpu_mem_str()}")
    return best_f1, tokenizer


# =============================================================================
# 5. Predict 1 model từ checkpoint (load → predict → del)
# =============================================================================
def predict_from_ckpt(model_name, ckpt_path, tokenizer, texts, cfg_task, device):
    """
    [OOM-2] Load model, predict, del ngay — không bao giờ giữ nhiều models.
    """
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=2, ignore_mismatched_sizes=True).to(device)
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.eval()

    ds     = HahaDataset(texts, None, tokenizer, cfg_task["max_len"])
    loader = DataLoader(ds, batch_size=cfg_task["batch_size"] * 2,
                        shuffle=False, num_workers=CFG["num_workers"])
    probs = []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            logits = model(**batch).logits
            probs.extend(torch.softmax(logits, -1).cpu().numpy())

    free_gpu(model)
    return np.array(probs)


# =============================================================================
# 6. Run one task
# =============================================================================
def run_task(
    task_name, train_df, test_df,
    text_fn, label2id, pos_label,
    cfg_task, model_name,
    device, use_fp16, task_tag,
):
    print("\n" + "=" * 65)
    print(f"  {task_name}  |  model: {model_name}")
    dist = train_df["tag"].value_counts().to_dict()
    print(f"  Train: {len(train_df)} samples  {dist}")
    print("=" * 65)

    id2label   = {v: k for k, v in label2id.items()}
    pos_idx    = label2id[pos_label]
    ckpt_dir   = Path(CFG["ckpt_dir"]) / task_tag
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    texts      = [text_fn(r) for _, r in train_df.iterrows()]
    labels     = [label2id[t] for t in train_df["tag"]]
    labels_arr = np.array(labels)
    test_texts = [text_fn(r) for _, r in test_df.iterrows()]

    skf        = StratifiedKFold(n_splits=CFG["folds"], shuffle=True,
                                 random_state=CFG["seed"])
    fold_f1s   = []
    oof_preds  = np.zeros(len(labels_arr), dtype=int)
    oof_probs  = np.zeros((len(labels_arr), 2))

    # Lưu (ckpt_path, tokenizer) thay vì model object
    fold_ckpts     = []
    fold_tokenizers = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(texts, labels), 1):
        print(f"\n── Fold {fold}/{CFG['folds']} "
              f"(train={len(tr_idx)}, val={len(val_idx)}) ──")

        ckpt_path = str(ckpt_dir / f"fold{fold}.pt")

        best_f1, tokenizer = train_fold(
            model_name, ckpt_path,
            [texts[i] for i in tr_idx], [labels[i] for i in tr_idx],
            [texts[i] for i in val_idx], [labels[i] for i in val_idx],
            cfg_task, pos_idx,
            device, use_fp16, fold,
        )
        fold_f1s.append(best_f1)
        fold_ckpts.append(ckpt_path)
        fold_tokenizers.append(tokenizer)

        # OOF probs — load best ckpt, predict val, del
        vp = predict_from_ckpt(
            model_name, ckpt_path, tokenizer,
            [texts[i] for i in val_idx],
            cfg_task, device,
        )
        oof_probs[val_idx] = vp
        oof_preds[val_idx] = vp.argmax(1)

    # ── OOF ──────────────────────────────────────────────────────────────────
    oof_f1 = f1_score(labels_arr, oof_preds,
                      pos_label=pos_idx, average="binary", zero_division=0)
    print("\n" + "=" * 65)
    print(f"  {task_name}")
    print(f"  OOF F1 ({pos_label}): {oof_f1:.4f}")
    print(f"  Fold F1s : {[round(f,4) for f in fold_f1s]}")
    print(f"  Mean±Std : {np.mean(fold_f1s):.4f} ± {np.std(fold_f1s):.4f}")
    print("=" * 65)
    print(classification_report(
        [id2label[l] for l in labels_arr],
        [id2label[p] for p in oof_preds],
        zero_division=0,
    ))

    # ── Ensemble test predict — 1 model at a time ─────────────────────────
    print(f"  Ensemble predict test ({len(test_texts)} samples), "
          f"{len(fold_ckpts)} models, 1-at-a-time...")
    test_probs_sum = np.zeros((len(test_texts), 2))
    for ckpt_path, tokenizer in zip(fold_ckpts, fold_tokenizers):
        p = predict_from_ckpt(model_name, ckpt_path, tokenizer,
                              test_texts, cfg_task, device)
        test_probs_sum += p
    test_preds = [id2label[i] for i in (test_probs_sum / len(fold_ckpts)).argmax(1)]
    print(f"  Test dist: {pd.Series(test_preds).value_counts().to_dict()}")

    return oof_f1, test_preds


# =============================================================================
# 7. Main
# =============================================================================
def main():
    set_seed(CFG["seed"])
    Path(CFG["output_dir"]).mkdir(parents=True, exist_ok=True)

    device   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_fp16 = torch.cuda.is_available()
    print(f"Device  : {device}  |  fp16: {use_fp16}  {gpu_mem_str()}")

    data_dir = Path(CFG["data_dir"])
    test_dir = Path(CFG["test_dir"])
    out_dir  = Path(CFG["output_dir"])

    # ── Load ──────────────────────────────────────────────────────────────────
    t1_trial    = pd.read_csv(data_dir / "task1_trial.tsv",     sep="\t").dropna(subset=["tag"])
    t1_dev_gold = pd.read_csv(test_dir / "task1_dev_gold.tsv",  sep="\t").dropna(subset=["tag"])
    t1_test     = pd.read_csv(test_dir / "task1_test.tsv",      sep="\t")

    t2_trial    = pd.read_csv(data_dir / "task2_trial.tsv",     sep="\t").dropna(subset=["tag"])
    t2_dev_gold = pd.read_csv(test_dir / "task2_dev_gold.tsv",  sep="\t").dropna(subset=["tag"])
    t2_test     = pd.read_csv(test_dir / "task2_test.tsv",      sep="\t")

    t1_train = pd.concat([t1_trial, t1_dev_gold], ignore_index=True)
    t2_train = pd.concat([t2_trial, t2_dev_gold], ignore_index=True)

    print(f"\nT1 train: {len(t1_train)}  {t1_train.tag.value_counts().to_dict()}")
    print(f"T2 train: {len(t2_train)}  {t2_train.tag.value_counts().to_dict()}")
    print(f"T1 test : {len(t1_test)}  |  T2 test: {len(t2_test)}\n")

    # ── Task 1 ────────────────────────────────────────────────────────────────
    t1_oof_f1, t1_test_preds = run_task(
        task_name  = "TASK 1 — Humor Detection",
        train_df   = t1_train,
        test_df    = t1_test,
        text_fn    = make_t1_text,
        label2id   = {"satirical": 0, "real": 1},
        pos_label  = "satirical",
        cfg_task   = CFG["t1"],
        model_name = CFG["model_t1"],
        device     = device,
        use_fp16   = use_fp16,
        task_tag   = "t1",
    )

    # ── Task 2 ────────────────────────────────────────────────────────────────
    t2_oof_f1, t2_test_preds = run_task(
        task_name  = "TASK 2 — LLM Humor Detection",
        train_df   = t2_train,
        test_df    = t2_test,
        text_fn    = make_t2_text,
        label2id   = {"machine": 0, "human": 1},
        pos_label  = "machine",
        cfg_task   = CFG["t2"],
        model_name = CFG["model_t2"],
        device     = device,
        use_fp16   = use_fp16,
        task_tag   = "t2",
    )

    # ── Save ──────────────────────────────────────────────────────────────────
    t1_sub = pd.DataFrame({"id": t1_test["id"].values, "tag": t1_test_preds})
    t2_sub = pd.DataFrame({"id": t2_test["id"].values, "tag": t2_test_preds})

    t1_path  = out_dir / "task1.tsv"
    t2_path  = out_dir / "task2.tsv"
    zip_path = out_dir / "submission.zip"

    t1_sub.to_csv(t1_path, sep="\t", index=False)
    t2_sub.to_csv(t2_path, sep="\t", index=False)
    with zipfile.ZipFile(zip_path, "w") as zf:
        zf.write(t1_path, arcname="task1.tsv")
        zf.write(t2_path, arcname="task2.tsv")

    # ── Summary ───────────────────────────────────────────────────────────────
    print()
    print("=" * 65)
    print("FINAL SUMMARY — v3.1")
    print("=" * 65)
    print(f"T1 ({CFG['model_t1']})")
    print(f"   train={len(t1_train)}  OOF F1={t1_oof_f1:.4f}")
    print(f"   test dist: {t1_sub['tag'].value_counts().to_dict()}")
    print()
    print(f"T2 ({CFG['model_t2']})")
    print(f"   train={len(t2_train)}  OOF F1={t2_oof_f1:.4f}")
    print(f"   test dist: {t2_sub['tag'].value_counts().to_dict()}")
    print(f"\nsubmission.zip → {zip_path}")


if __name__ == "__main__":
    main()

Device  : cuda  |  fp16: True  [GPU 0.1/14.6GB]

T1 train: 564  {'real': 282, 'satirical': 282}
T2 train: 362  {'machine': 181, 'human': 181}
T1 test : 600  |  T2 test: 550


  TASK 1 — Humor Detection  |  model: dccuchile/bert-base-spanish-wwm-cased
  Train: 564 samples  {'real': 282, 'satirical': 282}

── Fold 1/5 (train=451, val=113) ──


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.weight                   | MISSING    | 
classifier.bias                            | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not 

  F1E01 [GPU 0.6/14.6GB]:   0%|          | 0/29 [00:00<?, ?it/s]

  F1 E01  loss=0.6744  val_F1=0.7597 ✓  [GPU 2.6/14.6GB]


...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Test dist: {'machine': 277, 'human': 273}

FINAL SUMMARY — v3.1
T1 (dccuchile/bert-base-spanish-wwm-cased)
   train=564  OOF F1=0.8043
   test dist: {'real': 302, 'satirical': 298}

T2 (xlm-roberta-base)
   train=362  OOF F1=0.8462
   test dist: {'machine': 277, 'human': 273}

submission.zip → /kaggle/working/submission.zip
